# 09 - Model: Hybrid

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import load_npz, save_npz
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from sklearn.utils.sparsefuncs import inplace_csr_column_scale

warnings.filterwarnings("ignore")

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models" / "content_based"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## Load

In [2]:
tracks = pd.read_csv(PROCESSED_DIR / "content_tracks_engineered.csv")
feature_matrix = load_npz(MODEL_DIR / "engineered_feature_matrix.npz")

with open(MODEL_DIR / "engineered_feature_names.json", "r", encoding="utf-8") as file:
    feature_names = json.load(file)

print(f"Tracks: {tracks.shape}")
print(f"Feature matrix: {feature_matrix.shape}")

Tracks: (89740, 40)
Feature matrix: (89740, 4163)


## Weights

In [3]:
NUMERIC_WEIGHT = 1.0
CATEGORICAL_WEIGHT = 1.0
TEXT_TFIDF_WEIGHT = 2.0

feature_groups = pd.Series(feature_names).str.extract(r"^(numeric|categorical|text)__", expand=False)
feature_weights = np.select(
    [
        feature_groups.eq("numeric"),
        feature_groups.eq("categorical"),
        feature_groups.eq("text"),
    ],
    [NUMERIC_WEIGHT, CATEGORICAL_WEIGHT, TEXT_TFIDF_WEIGHT],
    default=1.0,
).astype(float)

weighted_feature_matrix = feature_matrix.tocsr(copy=True)
inplace_csr_column_scale(weighted_feature_matrix, feature_weights)
weighted_feature_matrix = normalize(weighted_feature_matrix, norm="l2", copy=False)

weight_config = {
    "numeric_weight": NUMERIC_WEIGHT,
    "categorical_weight": CATEGORICAL_WEIGHT,
    "text_tfidf_weight": TEXT_TFIDF_WEIGHT,
}
print(weighted_feature_matrix.shape)

(89740, 4163)


## Train

In [4]:
model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=51, n_jobs=-1)
model.fit(weighted_feature_matrix)

track_id_to_index = dict(zip(tracks["track_id"], tracks["content_index"]))

## Recommend

In [5]:
def recommend_by_track_id(track_id: str, top_n: int = 10) -> pd.DataFrame:
    if track_id not in track_id_to_index:
        raise ValueError(f"Unknown track_id: {track_id}")

    track_index = track_id_to_index[track_id]
    distances, indices = model.kneighbors(weighted_feature_matrix[track_index], n_neighbors=top_n + 1)

    rec_indices = indices.ravel()[1:]
    rec_distances = distances.ravel()[1:]

    columns = ["track_id", "track_name", "artists", "album_name", "track_genre", "popularity"]
    recommendations = tracks.iloc[rec_indices][columns].copy()
    recommendations["similarity_score"] = 1 - rec_distances
    return recommendations.reset_index(drop=True)

In [6]:
demo_seeds = pd.concat([tracks.iloc[[0]], tracks.sample(2, random_state=7)])

for _, seed in demo_seeds.iterrows():
    print(f"\n{seed['track_name']} - {seed['artists']} [{seed['track_genre']}]")
    display(recommend_by_track_id(seed["track_id"], top_n=5))


Comedy - Gen Hoshino [acoustic]


,track_id,track_name,artists,album_name,track_genre,popularity,similarity_score
0,6x15KrIZn8qZMAByvhPAZU,Los 90,Natos y Waor,Luna llena,spanish,57,0.739118
1,7LiGptffUbVAtbxzQnvBi9,OH NO!,Cody Lovaas;ROZES,Pull Out Couch,chill,57,0.736725
2,186AzR054q9nSWYSI3qr8D,Losing You,Christian Kuria,Borderline,chill,59,0.732376
3,4Nw7kywWurWS6ceinn1cHK,Baby Powder,Jenevieve,Division,chill,68,0.716352
4,7jIAttgQTpLDoNtykIQXjH,Blister In The Sun,Violent Femmes,Violent Femmes,acoustic,71,0.714812



Everybody's Somebody's Fool - Connie Francis [rockabilly]


,track_id,track_name,artists,album_name,track_genre,popularity,similarity_score
0,1CSLeVCXmetBh8IkTPMFdL,I'm a Believer,The Monkees,The Best of The Monkees,psych-rock,57,0.815436
1,6YS9gh6dxp4rYKWk5jjBDC,Jambalaya (On The Bayou),Fats Domino,Legends Of Rock n' Roll,rockabilly,53,0.775675
2,3G7tRC24Uh09Hmp1KZ7LQ2,I'm a Believer - 2006 Remaster,The Monkees,More of The Monkees (Deluxe Edition),psych-rock,74,0.774435
3,2aEeghgUcnu75tzcolFMfs,La Bamba - Single Version,Ritchie Valens,Ritchie Valens,rock-n-roll,66,0.756354
4,4OnqJ1ml4Jgr5AAKNrrYCD,Down in Mexico,The Coasters,The Coasters,rockabilly,54,0.755577



Wrath of Aeshma - From The Vastland [iranian]


,track_id,track_name,artists,album_name,track_genre,popularity,similarity_score
0,0xSAeGLEMNwWZw4eUHehpa,Temple of Daevas,From The Vastland,Temple of Daevas,iranian,0,0.961482
1,4keTweXLCnQUg2SjF6UFSX,The Joyful Wisdom,Garhelenth,Conspiracy Under The Black Flame,iranian,0,0.906773
2,164mALSzQg34QKAkPoS7bV,Chaos Above Order,Trivax,Sin,iranian,1,0.902876
3,1lgqemZ33dy7SBhJ3ZOcm8,Brain Scratch,Tension Prophecy,Asylum of Humanity,iranian,0,0.901258
4,0mfi1L8wTr5MsyTKrDo2nP,Ransom Note,Confess,Revenge at All Costs,iranian,9,0.894434


## Save

In [7]:
model_path = MODEL_DIR / "content_based_knn_tfidf_weighted_model.joblib"
weighted_matrix_path = MODEL_DIR / "tfidf_weighted_feature_matrix.npz"
weight_config_path = MODEL_DIR / "tfidf_weight_config.json"

joblib.dump(model, model_path)
save_npz(weighted_matrix_path, weighted_feature_matrix)

with open(weight_config_path, "w", encoding="utf-8") as file:
    json.dump(weight_config, file, indent=2)

print("Saved:", model_path, weighted_matrix_path, weight_config_path, sep="\n")

Saved:
/run/media/ashish_subedi/c287b5fb-f7e6-40ae-be97-12df427f3339/ml/sportify-recommendation/recommender/models/content_based/content_based_knn_tfidf_weighted_model.joblib
/run/media/ashish_subedi/c287b5fb-f7e6-40ae-be97-12df427f3339/ml/sportify-recommendation/recommender/models/content_based/tfidf_weighted_feature_matrix.npz
/run/media/ashish_subedi/c287b5fb-f7e6-40ae-be97-12df427f3339/ml/sportify-recommendation/recommender/models/content_based/tfidf_weight_config.json
